In [3]:
import pandas as pd
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv()
client = Groq(api_key=os.getenv('GROQ_API_KEY'))

# تحميل بيانات المقارنة اللي عملناها قبل كده
comparison_store = pd.read_csv(r'C:\Users\adham\Downloads\walmart-recruiting-store-sales-forecasting\Outputs\forecast_comparison_store40.csv')

In [4]:
def prepare_data_summary(df):
    """
    تحويل جدول المقارنة لنص مختصر يوصف الأنماط الأساسية
    عشان نبعته للـ AI بدل ما نبعت الجدول الخام كامل
    """
    worst_week = df.loc[df['error'].abs().idxmax()]
    total_actual = df['Weekly_Sales'].sum()
    total_forecast = df['yhat'].sum()
    mape = (df['error'].abs() / df['Weekly_Sales']).mean() * 100

    summary = f"""
Store 40 - Forecast Performance Summary:
- Total Actual Sales: {total_actual:,.0f}
- Total Forecasted Sales: {total_forecast:,.0f}
- Overall MAPE: {mape:.2f}%
- Worst forecast week: {worst_week['Date']} (Actual: {worst_week['Weekly_Sales']:,.0f}, Forecast: {worst_week['yhat']:,.0f}, Error: {worst_week['error']:,.0f})
- Number of weeks analyzed: {len(df)}
"""
    return summary

summary_text = prepare_data_summary(comparison_store)
print(summary_text)


Store 40 - Forecast Performance Summary:
- Total Actual Sales: 11,610,304
- Total Forecasted Sales: 11,567,608
- Overall MAPE: 3.76%
- Worst forecast week: 2012-09-07 (Actual: 1,088,248, Forecast: 962,264, Error: 125,985)
- Number of weeks analyzed: 12



In [8]:
def generate_ai_analysis(summary_text):
    """
    بعت التلخيص النصي لموديل Groq وخليه يرجع تحليل بشري القراءة
    """
    prompt = f"""You are a retail demand planning analyst. Based on the forecast performance summary below, write a concise business analysis anyone can understand (4-5 sentences) covering:
1. Overall forecast reliability
2. Likely reason behind the worst-performing week
3. One actionable recommendation for the demand planning team

Data:
{summary_text}
"""

    response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[{"role": "user", "content": prompt}],
    temperature=0.3,
    max_tokens=400
)

    return response.choices[0].message.content

analysis = generate_ai_analysis(summary_text)
print(analysis)

The forecast for Store 40 was very reliable overall, with an average absolute percentage error of just **3.8 %** across the 12‑week period, meaning the model was typically within a few percent of actual sales. The single week that stood out—**September 7, 2012**—saw a **13 % under‑forecast** (about 126 k units), which is far larger than the norm. This spike is most likely tied to an **unplanned promotion, special event, or supply‑chain disruption** that wasn’t captured in the historical data used to build the forecast.  

**Recommendation:** Integrate the store’s promotional and event calendar (and any known supply‑chain alerts) into the forecasting process, and flag weeks with planned promotions so the model can automatically adjust demand expectations. This will help prevent similar large‑error weeks and keep the overall MAPE low.
